# Домашнее задание 1: персональные top-10 рекомендации

**ФИО:** заполнить перед сдачей.

**Данные:** [Amazon Review Data 2018, Musical Instruments](https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/index.html),
5-core и метаданные товаров. Исходная выборка содержит 231 392 отзыва.
Авторы: Jianmo Ni, Jiacheng Li, Julian McAuley, EMNLP 2019.

Рекомендуем ранее не оценённые товары. Релевантность — оценка **4 или 5** в будущем периоде.
Отзыв — наблюдаемое взаимодействие, а не полная информация о покупках или показах.

Пять обязательных моделей: ItemKNN, iALS, EASE, FM с метаданными, iALS с собственным таргетом.
Дополнительно сравним Popularity и FM без метаданных.


In [ ]:
import os
from pathlib import Path

SEED = 42
N_THREADS = 2
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DEVICE = "cpu"

for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[variable] = str(N_THREADS)

In [ ]:
import json
import random
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from data import (
    SOURCE, FILES, download_data, read_reviews, clean_reviews,
    read_metadata, encode_ids, temporal_split, make_item_features,
)
from evaluation import prepare_evaluation
from experiment import tune_models, final_comparison
from models import ItemKNN, IALS, EASE, FM, CustomIALS, Popularity

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(N_THREADS)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 4)

display(pd.Series({name: version(name) for name in ["numpy", "scipy", "pandas", "torch", "matplotlib"]}, name="version"))

## 1. Загрузка и смысл полей

- `user_id` (`reviewerID`) — пользователь.
- `asin` — товар, единица рекомендации.
- `rating` (`overall`) — оценка 1–5.
- `timestamp` (`unixReviewTime`) — дата отзыва, далее переводится в UTC.
- `title`, `brand`, `category` — метаданные товара; `category` — последняя категория пути.

Читаем из архивов только нужные поля. Метаданные загружаем только для товаров из выбранного 5-core.
Все оценки сохраняются, включая низкие. В GitHub сами архивы не загружаются.

In [ ]:
display(pd.Series(FILES, name="download_url"))
download_data(DATA_DIR)

raw = read_reviews(DATA_DIR / "Musical_Instruments_5.json.gz")
display(raw.head())
print(f"Исходное число отзывов: {len(raw):,}")

## 2. EDA и подготовка

Сначала проверяем пропуски и повторы. Удаляем строки без ID, оценки или корректной даты,
а также оценки вне диапазона 1–5. Для повторных пар сохраняем **первый** отзыв:
поздняя оценка не должна подменять раннюю историю или становиться повторной рекомендацией.
Дополнительные k-core фильтры и случайную подвыборку не применяем.

In [ ]:
display(raw.isna().sum().to_frame("missing"))
display(pd.Series({
    "full_duplicates": int(raw.duplicated().sum()),
    "repeated_user_item": int(raw.duplicated(["user_id", "asin"]).sum()),
    "invalid_ratings": int((~raw.rating.between(1, 5) & raw.rating.notna()).sum()),
}, name="rows"))

cleaned = clean_reviews(raw)
valid_rows = raw.dropna(subset=["user_id", "asin", "rating", "timestamp"])
valid_rows = valid_rows[valid_rows.rating.between(1, 5)]
valid_rows = valid_rows[pd.to_datetime(valid_rows.timestamp, unit="s", utc=True, errors="coerce").notna()]
display(pd.DataFrame({
    "step": ["raw", "valid fields", "first user-item interaction"],
    "interactions": [len(raw), len(valid_rows), len(cleaned)],
}))

data, user_ids, item_ids = encode_ids(cleaned)
shape = (len(user_ids), len(item_ids))
items = read_metadata(DATA_DIR / "meta_Musical_Instruments.json.gz", item_ids)

assert 100_000 <= len(data) <= 2_000_000, "Итоговый объём не соответствует условию ДЗ."
assert min(shape) >= 1_000, "Нужно не менее 1000 пользователей и товаров."
display(data.head())

In [ ]:
user_activity = data.groupby("user").size()
item_popularity = data.groupby("item").size()
sparsity = 1 - len(data) / (shape[0] * shape[1])

display(pd.Series({
    "raw_interactions": len(raw),
    "clean_interactions": len(data),
    "users": shape[0],
    "items": shape[1],
    "sparsity": sparsity,
    "positive_fraction": (data.rating >= 4).mean(),
    "top_1_percent_item_share": item_popularity.nlargest(max(1, shape[1] // 100)).sum() / len(data),
}, name="value"))
display(pd.DataFrame({
    "user_activity": user_activity.describe(percentiles=[0.5, 0.9, 0.99]),
    "item_popularity": item_popularity.describe(percentiles=[0.5, 0.9, 0.99]),
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_activity, bins=40, log=True)
axes[0].set(title="Активность пользователей", xlabel="Число отзывов", ylabel="Число пользователей, log")
axes[1].hist(item_popularity, bins=40, log=True)
axes[1].set(title="Популярность товаров", xlabel="Число отзывов", ylabel="Число товаров, log")
plt.tight_layout()
plt.show()

top_items = item_popularity.nlargest(10).rename("reviews").to_frame().join(items)
display(top_items[["asin", "title", "brand", "category", "reviews"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
data.rating.value_counts().sort_index().plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Оценки", xlabel="Оценка", ylabel="Число отзывов")
monthly = data.set_index("timestamp").resample("MS").size()
axes[1].plot(monthly.index, monthly.values)
axes[1].set(title="Взаимодействия по месяцам", xlabel="Дата", ylabel="Число отзывов")
plt.tight_layout()
plt.show()
print("Период:", data.timestamp.min(), "—", data.timestamp.max())

In [ ]:
metadata_coverage = pd.DataFrame({
    "missing_fraction": items[["title", "brand", "category"]].eq("").mean(),
    "unique_nonempty": items[["title", "brand", "category"]].replace("", np.nan).nunique(),
})
display(metadata_coverage)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for field, axis in zip(["brand", "category"], axes):
    items[field].replace("", "UNKNOWN").value_counts().head(10).sort_values().plot.barh(ax=axis)
    axis.set(title=f"Топ-10: {field}", xlabel="Число товаров")
plt.tight_layout()
plt.show()

**Решения по подготовке.** Разреженные матрицы храним в CSR. Не удаляем товары из-за отсутствия
метаданных: FM получает unknown-коды, а все модели сохраняют одинаковый каталог.
Один пользователь–товар встречается в итоговых данных один раз. Низкие оценки остаются
отрицательным сигналом для iALS и FM, а ItemKNN и EASE строятся по оценкам 4–5.

Готовый 5-core сформирован авторами по всему периоду. Поэтому это сравнение на ретроспективно
отфильтрованной выборке, а не точная имитация нового сервиса. После дедупликации минимальные
степени могут стать меньше пяти. Бренд и категория считаются постоянными: исторических снимков
метаданных нет. Цену, sales rank, связи also-bought и тексты отзывов не используем.

## 3. Протокол оценки — фиксируем до подбора

- Общие временные границы: примерно 80% / 10% / 10% взаимодействий; одинаковые даты не разделяются.
- Validation: обучение на train. Финальный test: обучение заново на train + validation.
- Кандидаты: **все товары из текущей обучающей истории**, без семплирования каталога.
- Исключаем все ранее оценённые товары пользователя, независимо от оценки.
- Основной эксперимент — на известных пользователях и товарах. Cold start отдельно считаем ниже.
- Для пользователя нужен хотя бы один релевантный доступный товар и минимум 10 оставшихся кандидатов.
- Пользователи, кандидаты и правила для всех моделей одинаковы. Обучающие таргеты могут различаться,
  но релевантность всегда равна `rating >= 4`.

**Основная метрика — NDCG@10:** учитывает, насколько высоко стоят релевантные товары.
**Вторая — Recall@10:** доля найденных релевантных товаров среди доступных релевантных в будущем периоде.
Обе метрики усредняются по пользователям, а не по взаимодействиям.

$$DCG@10 = \sum_{j=1}^{10}\frac{rel_j}{\log_2(j+1)},\quad
NDCG@10 = \frac{DCG@10}{IDCG@10},\quad
Recall@10 = \frac{|Rec_{10}(u)\cap R_u|}{|R_u|}.$$

Пользователи без доступных релевантных товаров исключаются из обеих метрик. Ненаблюдаемый товар
не обязательно плохой, но в offline-оценке за него нельзя получить положительное попадание.
Параметры выбираем только по validation; test не используем для исправления настроек.

In [ ]:
train, valid, test = temporal_split(data)
split_summary = []
for name, part in [("train", train), ("validation", valid), ("test", test)]:
    split_summary.append({
        "split": name, "interactions": len(part), "users": part.user.nunique(),
        "items": part.item.nunique(), "positive_interactions": int((part.rating >= 4).sum()),
        "start": part.timestamp.min(), "end": part.timestamp.max(),
    })
display(pd.DataFrame(split_summary).set_index("split"))
assert train.timestamp.max() < valid.timestamp.min()
assert valid.timestamp.max() < test.timestamp.min()

train_valid = pd.concat([train, valid], ignore_index=True)

In [ ]:
coverage = []
for name, history, future in [("validation", train, valid), ("test", train_valid, test)]:
    seen, candidates, truth = prepare_evaluation(history, future, shape)
    cold_users = ~future.user.isin(history.user.unique())
    cold_items = ~future.item.isin(candidates)
    coverage.append({
        "split": name,
        "future_users": future.user.nunique(),
        "evaluated_users": len(truth),
        "excluded_users": future.user.nunique() - len(truth),
        "cold_user_interactions": int(cold_users.sum()),
        "cold_item_interactions": int(cold_items.sum()),
        "positive_interactions": int((future.rating >= 4).sum()),
        "evaluated_positive_pairs": sum(len(values) for values in truth.values()),
        "catalog_size": len(candidates),
    })
display(pd.DataFrame(coverage).set_index("split"))

# Один и тот же обучающий словарь метаданных для всех validation-экспериментов.
item_features = make_item_features(items, train, shape[0])
print("FM: item ID, brand, category; shape =", item_features.shape)

## 4. Модели и формулы

### ItemKNN

Пусть $X_{ui}=1[r_{ui}\geq4]$, а $x_i$ — столбец товара.
$$s_{ij}=\frac{x_i^\top x_j}{\|x_i\|\|x_j\|+shrink},\quad
score(u,i)=\sum_j X_{uj}S_{ji}.$$
Диагональ $S$ обнуляем; в каждой строке сохраняем $k$ ближайших соседей.
Таким образом каждый товар истории голосует за свои похожие товары.

### iALS

$$\min_{U,V}\sum_{u,i}c_{ui}(p_{ui}-U_u^\top V_i)^2
+\lambda(\|U\|_F^2+\|V\|_F^2).$$
Таргет $p_{ui}=1[r_{ui}\geq4]$ для наблюдаемой пары и 0 иначе.
Confidence $c_{ui}=1+\alpha$ для **любой наблюдаемой пары**, включая низкие оценки,
и 1 для ненаблюдаемой. Отличаем известную низкую оценку от отсутствия данных.
Чередуем аналитические обновления:
$$U_u=(V^\top C_uV+\lambda I)^{-1}V^\top C_up_u.$$
В коде решаем систему, не вычисляя обратную матрицу. Обновление $V$ симметрично.

### EASE

$$\min_B \|X-XB\|_F^2+\lambda\|B\|_F^2,\quad diag(B)=0.$$
$$P=(X^\top X+\lambda I)^{-1},\quad B_{ij}=-P_{ij}/P_{jj},\quad B_{ii}=0.$$
Предсказания: $XB$. Здесь нужна плотная матрица размера $I\times I$.

### FM с метаданными

Вектор признаков содержит четыре категориальных поля: **user ID, item ID, бренд, категория**.
Бренд отражает предпочтение производителя, категория — тип инструмента или аксессуара.
Словари категорий строятся по обучающим товарам; пропуски получают отдельный код.

$$\hat y(x)=w_0+\sum_jw_jx_j+\frac12\sum_f
\left[\left(\sum_jv_{jf}x_j\right)^2-\sum_jv_{jf}^2x_j^2\right].$$
Оптимизируем binary cross-entropy с L2-регуляризацией через Adam.
Наблюдаемые оценки 4–5 имеют label 1, оценки 1–3 — label 0.
На каждый положительный обучающий пример добавляем два случайных ненаблюдаемых товара
того же пользователя с label 0. Из семплирования исключаем **только обучающую историю**,
не заглядывая в validation/test. Отрицательные пары пересемплируются каждую эпоху.
Сравнение с FM только на ID помогает оценить вклад метаданных.

При ранжировании используем score до sigmoid: монотонное преобразование не меняет порядок.
Кешируем часть FM, зависящую только от товара; результат совпадает с исходной FM-формулой.

### iALS с собственным таргетом

Гипотеза: высокая недавняя оценка сильнее отражает текущий интерес.

$$p^{custom}_{ui}=1[r_{ui}\geq4]\cdot\frac{r_{ui}-3}{2}
\cdot 2^{-(T-t_{ui})/H}.$$

$T$ — максимальная дата **в текущей обучающей истории**, $H$ — период полураспада в днях.
Для ненаблюдаемых пар таргет 0. Оценка 4 получает половину силы оценки 5 до затухания.
Изменяется именно целевая матрица $p$, а confidence и ALS-обновления остаются теми же.
У базовой и собственной версии одинаковая сетка факторов и регуляризации; $\alpha=20$ у обеих.
Общий test-критерий релевантности остаётся бинарным: оценка 4–5.

В классе также есть `use_rating=False` и `half_life_days=None` для изучения компонентов
таргета. При отключении обоих компонентов получается в точности базовый iALS.
Popularity — дополнительный baseline: число положительных обучающих отзывов товара.

## 5. Подбор на validation

Небольшие сетки выбраны заранее. Для iALS рассматриваем 32/64 фактора и регуляризацию 0.1/1.0,
для custom iALS добавляем полураспад 365/1095 дней. У обеих версий 10 ALS-итераций и alpha=20.
В FM перебираем 16/32 фактора; остальные настройки фиксированы, одинаковые для FM с метаданными и без.
ItemKNN: 50/100 соседей и shrink 0/10. EASE: регуляризация 100/1000.
Это небольшой стартовый поиск, а не доказательство оптимальности каждой модели.

Все испытания выполняются последовательно, чтобы не держать несколько EASE в памяти.
При одинаковом NDCG выбирается первая конфигурация в списке. После validation настройки фиксируются.

In [ ]:
als_configs = [
    {"factors": factors, "reg": reg, "alpha": 20.0, "iterations": 10, "seed": SEED}
    for factors in [32, 64] for reg in [0.1, 1.0]
]
fm_configs = [
    {"factors": factors, "lr": 0.005, "reg": 1e-5, "epochs": 5,
     "batch_size": 2048, "negative_ratio": 2, "device": DEVICE, "seed": SEED}
    for factors in [16, 32]
]

specs = {
    "Popularity": (Popularity, [{}]),
    "ItemKNN": (ItemKNN, [{"k": k, "shrink": shrink} for k in [50, 100] for shrink in [0.0, 10.0]]),
    "iALS": (IALS, als_configs),
    "EASE": (EASE, [{"reg": reg} for reg in [100.0, 1000.0]]),
    "FM": (FM, [{**params, "use_metadata": True} for params in fm_configs]),
    "custom iALS": (CustomIALS, [
        {**params, "use_rating": True, "half_life_days": days}
        for params in als_configs for days in [365, 1095]
    ]),
    "FM IDs": (FM, [{**params, "use_metadata": False} for params in fm_configs]),
}
display(pd.Series({name: len(configs) for name, (_, configs) in specs.items()}, name="trials"))

In [ ]:
validation_results, best_params = tune_models(specs, train, valid, shape, item_features)
display(validation_results.sort_values(["model", "NDCG@10"], ascending=[True, False]))
display(pd.Series(best_params, name="chosen_parameters"))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
validation_results.to_csv(RESULTS_DIR / "validation.csv", index=False)
with (RESULTS_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump({
        "seed": SEED, "threads": N_THREADS, "source": SOURCE,
        "shape": shape, "interactions": len(data),
        "train_end": str(train.timestamp.max()), "valid_end": str(valid.timestamp.max()),
        "search_spaces": {name: configs for name, (_, configs) in specs.items()},
        "best_params": best_params,
        "versions": {name: version(name) for name in ["numpy", "scipy", "pandas", "torch", "matplotlib"]},
    }, file, ensure_ascii=False, indent=2)

## 6. Финальный test

Переобучаем выбранные конфигурации с нуля на train + validation. Словарь метаданных теперь также
использует train + validation; кандидаты — товары, известные к началу test. Все модели получают
один набор тестовых пользователей. Test применяется только для этой финальной таблицы.
Время оценки включает all-item scoring, маскирование, ранжирование и подсчёт метрик.

In [ ]:
final_item_features = make_item_features(items, train_valid, shape[0])
test_results, per_user_results = final_comparison(
    specs, best_params, train_valid, test, shape, final_item_features
)
display(test_results.reset_index(drop=True))
test_results.to_csv(RESULTS_DIR / "test.csv", index=False)
pd.concat(per_user_results, names=["model", "user"]).to_csv(RESULTS_DIR / "test_per_user.csv")

In [ ]:
table = test_results.set_index("model")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for metric, axis in zip(["NDCG@10", "Recall@10"], axes):
    table[metric].sort_values().plot.barh(ax=axis)
    axis.set(title=f"Test: {metric}", xlabel=metric)
plt.tight_layout()
plt.show()

print("Лучший NDCG@10:", table["NDCG@10"].idxmax())
display(pd.DataFrame({
    "FM minus FM IDs": table.loc["FM", ["NDCG@10", "Recall@10"]] - table.loc["FM IDs", ["NDCG@10", "Recall@10"]],
    "custom iALS minus iALS": table.loc["custom iALS", ["NDCG@10", "Recall@10"]] - table.loc["iALS", ["NDCG@10", "Recall@10"]],
}))

## 7. Выводы — заполнить после полного запуска

1. Какая из пяти обязательных моделей получила лучший NDCG@10? Совпадает ли лидер по Recall@10?
   Приведите значения и выбранные гиперпараметры из таблицы.
2. Насколько модели превосходят Popularity? Как качество соотносится со временем обучения и оценки?
3. Как отличаются FM с метаданными и FM IDs? Какой вывод поддерживают полученные числа?
4. Как отличается custom iALS от базового? Помогает ли учёт силы оценки и давности?
   Параметры каждой модели подбирались отдельно: разница лучших результатов не изолирует влияние
   каждого компонента таргета. Для такого вывода нужны дополнительные сравнения при равных настройках.
5. Какую долю будущих пользователей и положительных взаимодействий мы исключили? На кого
   распространяются результаты эксперимента?

Ограничения: ретроспективный 5-core, cold start вне основного сравнения, небольшая сетка параметров,
один временной split и seed, отсутствие показов и негативов с достоверным отсутствием интереса,
отсутствие исторических метаданных. Значимость небольших различий отдельно не проверялась.
Нельзя по offline-метрикам утверждать, что модель увеличит реальные покупки.

Перед сдачей: заполнить ФИО и выводы, сохранить выполненный notebook, проверить отсутствие данных
в git и опубликовать открытый репозиторий со ссылкой на источник данных.